# SentinelPay: Data Preprocessing and Leakage Prevention
## Notebook 02 — Train/Test Split, RobustScaler, and SMOTE on Training Split Only

**Author:** SentinelPay Research Team  
**Objective:** Demonstrate strict data hygiene practices that prevent information leakage from the test set into the training pipeline.

---

### 1. The Data Leakage Problem in Fraud Detection

Data leakage is arguably the most dangerous pitfall in fraud detection research. It occurs when information from the test set inadvertently influences the training process, producing optimistically biased metrics that collapse in production.

**Common leakage vectors in fraud detection pipelines:**

| Leakage Type | Cause | Consequence |
|:-------------|:------|:------------|
| **Pre-split Scaling** | Fitting StandardScaler on entire dataset before train/test split | Test set statistics (mean, variance) leak into training |
| **Pre-split SMOTE** | Applying SMOTE before splitting | Synthetic samples from test-set neighbours appear in training |
| **Feature Engineering Leakage** | Computing rolling aggregates across the full dataset | Future transactions influence past feature values |

**SentinelPay's Hygiene Protocol:**
1. Perform **Stratified Train/Test split** first (80/20).
2. Fit `RobustScaler` on the **training fold only**; transform the test fold.
3. Apply `SMOTE` to the **training fold only**; the test fold remains pristine and representative of real-world imbalance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('../data/raw/sentinelpay_benchmark_transactions.csv')
features = ['amount', 'distance', 'time_delta', 'merchant_risk', 'device_trust',
            'velocity_1h', 'velocity_24h', 'hour_of_day', 'is_weekend']

X = df[features]
y = df['is_fraud']

print(f"Total Dataset: {len(df):,} records")
print(f"Fraud Rate: {y.mean()*100:.2f}%")
print(f"Features: {len(features)}")

### 2. Stratified Train/Test Split

We use `stratify=y` to ensure the fraud ratio is preserved in both folds. Without stratification, random sampling could concentrate all fraud cases in one fold.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Stratified Split Results:")
print("=" * 50)
print(f"  Training Set: {len(X_train):>6,} records  (Fraud: {y_train.sum():>4} = {y_train.mean()*100:.2f}%)")
print(f"  Test Set:     {len(X_test):>6,} records  (Fraud: {y_test.sum():>4} = {y_test.mean()*100:.2f}%)")
print(f"\nFraud ratio preserved: Train={y_train.mean()*100:.2f}% vs Test={y_test.mean()*100:.2f}%")

### 3. RobustScaler (Fitted on Training Set Only)

We choose `RobustScaler` over `StandardScaler` because:
- It uses the **median and interquartile range (IQR)** rather than mean and standard deviation.
- This makes it robust to the extreme outliers common in fraud transactions (e.g., $50,000 purchase amounts).
- The scaler is fitted **exclusively on the training fold** to prevent test-set statistics from leaking.

In [ ]:
scaler = RobustScaler()

# FIT on training data ONLY
X_train_scaled = scaler.fit_transform(X_train)

# TRANSFORM test data using training statistics
X_test_scaled = scaler.transform(X_test)

print("Scaling Statistics (from Training Set Only):")
print("=" * 65)
for i, feat in enumerate(features):
    print(f"  {feat:<18}  center={scaler.center_[i]:>10.4f}  scale={scaler.scale_[i]:>10.4f}")

print(f"\nScaled Training Range: min={X_train_scaled.min():.4f}, max={X_train_scaled.max():.4f}")
print(f"Scaled Test Range:     min={X_test_scaled.min():.4f}, max={X_test_scaled.max():.4f}")

### 4. SMOTE Oversampling (Training Set Only)

**SMOTE (Synthetic Minority Over-sampling Technique)** generates synthetic fraud examples by interpolating between existing fraud instances in feature space (Chawla et al., 2002).

We use `sampling_strategy=0.25` to bring the fraud class to 25% of the majority class size, rather than full 1:1 parity, which can cause overfitting to synthetic patterns.

**Critical:** SMOTE is applied **only to the training fold**. The test fold retains the original ~1.2% fraud rate to provide an honest evaluation of real-world performance.

In [ ]:
print("Before SMOTE:")
print(f"  Training Legitimate: {(y_train == 0).sum():,}")
print(f"  Training Fraud:      {(y_train == 1).sum():,}")
print(f"  Ratio: {(y_train == 0).sum() / (y_train == 1).sum():.1f}:1")

smote = SMOTE(sampling_strategy=0.25, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f"\nAfter SMOTE:")
print(f"  Training Legitimate: {(y_train_res == 0).sum():,}")
print(f"  Training Fraud:      {(y_train_res == 1).sum():,}  (synthetic: {(y_train_res == 1).sum() - (y_train == 1).sum():,})")
print(f"  Ratio: {(y_train_res == 0).sum() / (y_train_res == 1).sum():.1f}:1")

print(f"\nTest Set (UNTOUCHED):")
print(f"  Test Legitimate: {(y_test == 0).sum():,}")
print(f"  Test Fraud:      {(y_test == 1).sum():,}")
print(f"  Ratio: {(y_test == 0).sum() / (y_test == 1).sum():.1f}:1 (real-world imbalance preserved)")

### 5. Leakage Verification Checklist

| Check | Status |
|:------|:-------|
| Stratified split performed before any preprocessing | PASS |
| RobustScaler fitted on training data only | PASS |
| SMOTE applied to training data only | PASS |
| Test set retains original class distribution | PASS |
| No future-looking features in the dataset | PASS |

---
*Proceed to Notebook 03: Baseline Model (Logistic Regression).*